# Lab 37: Building a Neural Network from Scratch

**Duration:** 3 hours 30 minutes. **Level:** intermediate. **Work mode:** pairs or individual.

| Student name | Station no. | Date |
|---|---|---|
| | | |

## 1. Overview
In Part 2 you used neural networks as black boxes, and in Lab 31 you trained one with PyTorch in a few lines. In this lab you open the box. You build a small neural network with **only NumPy**: one neuron, then a network with a hidden layer, the **forward pass** that makes a prediction, the **loss** that measures the mistake, and **backpropagation**, which finds how each weight must change. You check your gradients against a numerical test and against PyTorch's automatic differentiation, train the network on a problem that a straight line cannot solve (XOR), and then on two curved half-moons.

**By the end of this lab you can:**
1. Explain what a neuron, a layer, an activation function and a weight are.
2. Write the forward pass of a two-layer network in NumPy and check the shapes of every array.
3. Calculate the loss, and explain what a gradient is with a numerical test.
4. Implement backpropagation, and prove it right with a gradient check and with PyTorch.
5. Train the network with gradient descent and see how the number of hidden neurons and the learning rate change the result.

**How this notebook works.** Run a cell with **Shift + Enter**, from the top. Type your answers in the cells that say *Your answers*. Pictures and live video appear under the cell that makes them. Press **Ctrl + S** often.

> **Classification of this lab.** *Training from scratch* with NumPy. The network is small enough to run in a second on the CPU.

## 2. Background

**A neuron.** A neuron takes some numbers `x`, multiplies each one by a **weight** `w`, adds a **bias** `b`, and passes the sum through an **activation function**:

`output = activation(w1*x1 + w2*x2 + ... + b)`

Without the activation, a stack of neurons is just one big straight-line calculation. The activation adds the bend that lets a network learn curves.

| Activation | Formula | Shape |
|---|---|---|
| sigmoid | 1 / (1 + e^-z) | squeezes any number into 0 to 1 (used for the output of a yes/no answer) |
| tanh | (e^z - e^-z) / (e^z + e^-z) | squeezes into -1 to 1 |
| ReLU | max(0, z) | zero for negatives, straight for positives (the most used) |

**A layer** is many neurons side by side. With NumPy, a whole layer is one matrix product: `a = activation(X @ W + b)`. If `X` has shape (N, 2) and the layer has 4 neurons, then `W` has shape (2, 4), `b` has shape (4,) and `a` has shape (N, 4).

**The forward pass.** The data goes through the layers to a prediction.

**The loss.** One number for the mistake. For a yes/no answer with a sigmoid output `p` and the true label `y` (0 or 1) the **binary cross entropy** is `-[y log p + (1 - y) log(1 - p)]`, averaged over all examples. It is small when `p` is close to `y`.

**The gradient.** For every weight, the gradient says: *if I make this weight a little bigger, how much does the loss change?* If the gradient is positive, make the weight smaller. **Gradient descent** does this for all weights, a small step each time: `w = w - learning_rate * gradient`.

**Backpropagation** is the way to calculate all the gradients at once, using the chain rule, from the output back to the input. For our network:

| Step | Formula |
|---|---|
| Output error | `dz2 = (p - y) / N` |
| Gradient of the output layer | `dW2 = a1.T @ dz2`, `db2 = sum(dz2)` |
| Pass the error back | `da1 = dz2 @ W2.T`, `dz1 = da1 * (1 - a1**2)` (the slope of tanh) |
| Gradient of the hidden layer | `dW1 = X.T @ dz1`, `db1 = sum(dz1)` |

**Gradient check.** A mistake in backpropagation does not crash the program: the network just learns badly. So you compare your gradient with a slow but simple one: change one weight by a tiny amount `e` and measure the change of the loss, `(loss(w + e) - loss(w - e)) / (2e)`.

**Change one thing at a time.** If you break the notebook, close it and run `import labview; labview.reset(37)` in a cell of a new notebook (File, New, Notebook) for a fresh copy.

---
# Part A: Prepare (10 min)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(0)
print("numpy", np.__version__)

---
# Part B: One Neuron (30 min)

### Step 1: Calculate One Neuron by Hand
A neuron has two inputs. Calculate it on paper first: `z = 2*0.5 + 3*(-1.0) + 0.5`. Then run the cell and compare.

In [ ]:
x = np.array([2.0, 3.0])
w = np.array([0.5, -1.0])
b = 0.5

z = np.dot(w, x) + b
print("z (before the activation):", z)
print("sigmoid:", 1 / (1 + np.exp(-z)))
print("tanh   :", np.tanh(z))
print("relu   :", max(0.0, z))

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Your hand calculation of z and the result of the cell:
- Which activation gives a number between 0 and 1:

### Step 2: The Shape of the Activations
The picture shows the three activation functions.

In [ ]:
z = np.linspace(-5, 5, 200)
plt.figure(figsize=(6, 3.2))
plt.plot(z, 1 / (1 + np.exp(-z)), label="sigmoid")
plt.plot(z, np.tanh(z), label="tanh")
plt.plot(z, np.maximum(0, z), label="ReLU")
plt.ylim(-1.5, 2)
plt.legend()
plt.grid(alpha=0.3)
plt.show()

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- What happens to all three when z is very negative:
- Why is the sigmoid good for the last layer of a yes/no network:

---
# Part C: The Forward Pass of a Network (35 min)

### Step 1: The Problem That a Line Cannot Solve
XOR: the answer is 1 when the two inputs are **different**. No straight line separates the 1s from the 0s, so a single neuron fails. A network with one hidden layer can solve it.

In [ ]:
X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=float)
y = np.array([[0], [1], [1], [0]], dtype=float)
plt.figure(figsize=(3.4, 3.2))
plt.scatter(X[:, 0], X[:, 1], c=y[:, 0], cmap="coolwarm", s=140, edgecolor="k")
plt.title("XOR: red = 1, blue = 0")
plt.show()

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Try to draw one straight line on paper that puts both red points on one side. Can you:
- How many hidden neurons do you think you need:

### Step 2: The Network
The class holds the weights and has the forward pass. `H` is the number of hidden neurons. The weights start as small random numbers. The cell prints the **shape of every array**, and the predictions of the untrained network.

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))


class TinyNet:
    def __init__(self, inputs=2, hidden=4, seed=0):
        rng = np.random.default_rng(seed)
        self.W1 = rng.normal(0, 1.0, (inputs, hidden))
        self.b1 = np.zeros(hidden)
        self.W2 = rng.normal(0, 1.0, (hidden, 1))
        self.b2 = np.zeros(1)

    def forward(self, X):
        self.X = X
        self.a1 = np.tanh(X @ self.W1 + self.b1)          # hidden layer
        self.p = sigmoid(self.a1 @ self.W2 + self.b2)     # output: a number between 0 and 1
        return self.p

    def loss(self, y):
        p = np.clip(self.p, 1e-9, 1 - 1e-9)
        return float(-np.mean(y * np.log(p) + (1 - y) * np.log(1 - p)))


net = TinyNet(hidden=4)
p = net.forward(X)
print("shapes: X", X.shape, "| W1", net.W1.shape, "| b1", net.b1.shape, "| a1", net.a1.shape, "| W2", net.W2.shape, "| p", p.shape)
print("untrained predictions:", p.ravel().round(2), "| true:", y.ravel())
print("loss:", round(net.loss(y), 3), "(a coin toss gives 0.693)")
print("numbers (weights) in the network:", net.W1.size + net.b1.size + net.W2.size + net.b2.size)

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- The shapes printed:
- Number of weights and biases in the network:
- Is the untrained network right? What loss does it have compared with 0.693:

---
# Part D: The Gradient (40 min)

### Step 1: A Gradient by Measuring
Take one weight, `W1[0, 0]`. Make it a tiny bit bigger and a tiny bit smaller, and see how the loss changes. The ratio is the **numerical gradient**: slow, but you can trust it.

In [ ]:
def numerical_gradient(net, X, y, array, index, e=1e-5):
    old = array[index]
    array[index] = old + e
    net.forward(X)
    up = net.loss(y)
    array[index] = old - e
    net.forward(X)
    down = net.loss(y)
    array[index] = old
    net.forward(X)
    return (up - down) / (2 * e)


g = numerical_gradient(net, X, y, net.W1, (0, 0))
print("numerical gradient of W1[0, 0]:", round(g, 6))
print("meaning: if W1[0, 0] grows by 0.01, the loss changes by about", round(g * 0.01, 6))

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- The gradient of W1[0, 0]:
- In which direction must you change this weight to make the loss smaller:

### Step 2: Backpropagation
The method `backward` calculates **all** gradients at once with the formulas of the Background. Add it to the class (the cell adds it to `TinyNet`), then compare with the numerical gradient for **every** weight. If the biggest difference is smaller than `1e-6`, your backpropagation is right.

In [ ]:
def backward(self, y):
    N = len(y)
    dz2 = (self.p - y) / N                      # output error
    self.dW2 = self.a1.T @ dz2
    self.db2 = dz2.sum(axis=0)
    da1 = dz2 @ self.W2.T                       # pass the error back
    dz1 = da1 * (1 - self.a1 ** 2)              # times the slope of tanh
    self.dW1 = self.X.T @ dz1
    self.db1 = dz1.sum(axis=0)


TinyNet.backward = backward

net.forward(X)
net.backward(y)
worst = 0.0
for name, array, grad in (("W1", net.W1, net.dW1), ("b1", net.b1, net.db1), ("W2", net.W2, net.dW2), ("b2", net.b2, net.db2)):
    for index in np.ndindex(array.shape):
        worst = max(worst, abs(numerical_gradient(net, X, y, array, index) - grad[index]))
print("Biggest difference between backpropagation and the measurement:", worst)
print("Backpropagation is", "RIGHT" if worst < 1e-6 else "WRONG")

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- The biggest difference:
- Why is backpropagation used in practice and not the numerical gradient (count the forward passes the numerical way needs for this network):

---
# Part E: Train It (45 min)

### Step 1: Gradient Descent on XOR
The loop is: forward, loss, backward, then move every weight a small step against its gradient. The cell trains for 3000 steps and prints the loss now and then.

In [ ]:
def train(net, X, y, steps=3000, lr=0.5, every=500):
    history = []
    for step in range(steps):
        net.forward(X)
        history.append(net.loss(y))
        net.backward(y)
        net.W1 -= lr * net.dW1
        net.b1 -= lr * net.db1
        net.W2 -= lr * net.dW2
        net.b2 -= lr * net.db2
        if every and step % every == 0:
            print(f"step {step:>5}: loss {history[-1]:.4f}")
    net.forward(X)
    history.append(net.loss(y))
    return history


net = TinyNet(hidden=4, seed=1)
history = train(net, X, y)
print("loss at the end:", round(history[-1], 4))
print("predictions:", net.forward(X).ravel().round(3), "| true:", y.ravel())
print("accuracy:", ((net.p > 0.5) == y).mean() * 100, "%")

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- The loss at the start and at the end:
- The four predictions:
- Is XOR solved? What did the hidden layer add:

### Step 2: Look at the Loss
A healthy training: the loss falls fast at first and then slowly.

In [ ]:
plt.figure(figsize=(5, 3))
plt.plot(history)
plt.xlabel("step")
plt.ylabel("loss")
plt.grid(alpha=0.3)
plt.show()

### Step 3: Change One Thing: the Learning Rate and the Hidden Neurons
Train the network for several settings and compare the final loss.

In [ ]:
print("learning rate (4 hidden neurons):")
for lr in (0.01, 0.1, 0.5, 5.0, 50.0):
    h = train(TinyNet(hidden=4, seed=1), X, y, steps=3000, lr=lr, every=0)
    print(f"  lr {lr:<5} final loss {h[-1]:.4f}")
print("hidden neurons (lr 0.5):")
for hidden in (1, 2, 3, 8):
    h = train(TinyNet(hidden=hidden, seed=1), X, y, steps=3000, lr=0.5, every=0)
    print(f"  {hidden} neurons final loss {h[-1]:.4f}")

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Which learning rate was too small and which too big (look at the loss of 50):
- Can a network with 1 hidden neuron solve XOR? Why not:
- Which number of hidden neurons is the smallest that works well:

---
# Part F: Check with PyTorch (30 min)

### Step 1: Same Network, Automatic Gradients
PyTorch calculates gradients automatically (**autograd**). The cell builds the same network in PyTorch from the **same starting weights**, computes the loss, calls `loss.backward()`, and compares the gradients with yours. They must agree.

In [ ]:
import torch

net = TinyNet(hidden=4, seed=3)
net.forward(X)
net.backward(y)

W1, b1, W2, b2 = (torch.tensor(a, dtype=torch.float64, requires_grad=True) for a in (net.W1, net.b1, net.W2, net.b2))
Xt, yt = torch.tensor(X), torch.tensor(y)
p = torch.sigmoid(torch.tanh(Xt @ W1 + b1) @ W2 + b2)
loss = torch.nn.functional.binary_cross_entropy(p, yt)
loss.backward()

print("loss: numpy", round(net.loss(y), 6), "| torch", round(loss.item(), 6))
for name, mine, theirs in (("W1", net.dW1, W1.grad), ("b1", net.db1, b1.grad), ("W2", net.dW2, W2.grad), ("b2", net.db2, b2.grad)):
    print(f"{name}: biggest difference {np.abs(mine - theirs.numpy()).max():.2e}")

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- The biggest differences:
- What does autograd do for you that you wrote by hand in Part D:

---
# Part G: Curved Data, Two Half-Moons (40 min)

### Step 1: The Data
`make_moons` makes two curved groups of points with noise. A straight line makes mistakes on them. The cell uses 70 % for training and 30 % for the test.

In [ ]:
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split

Xm, ym = make_moons(n_samples=400, noise=0.1, random_state=0)
Xm_train, Xm_test, ym_train, ym_test = train_test_split(Xm, ym.reshape(-1, 1).astype(float), test_size=0.3, random_state=0, stratify=ym)
plt.figure(figsize=(4, 3.4))
plt.scatter(Xm[:, 0], Xm[:, 1], c=ym, cmap="coolwarm", s=12)
plt.show()

### Step 2: Train and Draw the Decision Boundary
The network learns to draw a curved line between the groups. The coloured background shows what the network answers at every point.

In [ ]:
def boundary(net, X, y, title):
    xs, ys = np.meshgrid(np.linspace(X[:, 0].min() - 0.5, X[:, 0].max() + 0.5, 150), np.linspace(X[:, 1].min() - 0.5, X[:, 1].max() + 0.5, 150))
    grid = np.c_[xs.ravel(), ys.ravel()]
    plt.contourf(xs, ys, net.forward(grid).reshape(xs.shape), levels=20, cmap="coolwarm", alpha=0.6)
    plt.scatter(X[:, 0], X[:, 1], c=y.ravel(), cmap="coolwarm", s=12, edgecolor="k", linewidth=0.3)
    plt.title(title)


results = {}
plt.figure(figsize=(11, 3.2))
for k, hidden in enumerate((1, 3, 16)):
    net = TinyNet(hidden=hidden, seed=0)
    train(net, Xm_train, ym_train, steps=4000, lr=0.5, every=0)
    net.forward(Xm_test)
    test_acc = ((net.p > 0.5) == ym_test).mean()
    results[hidden] = test_acc
    plt.subplot(1, 3, k + 1)
    boundary(net, Xm_train, ym_train, f"{hidden} hidden: test {test_acc * 100:.0f} %")
plt.show()
print({h: f"{a * 100:.0f} %" for h, a in results.items()})

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- The test accuracy for 1, 3 and 16 hidden neurons:
- What does the line of the 1-neuron network look like, and why:
- Does 16 neurons always beat 3? What could go wrong with too many (think of Lab 31):

---
# Part H: Make It Your Own and Present (20 min)
Choose **one**:

- Change the activation of the hidden layer from tanh to ReLU. You must change `forward` and the slope in `backward` (the slope of ReLU is 1 for positive and 0 for negative). Run the gradient check again.
- Add a second hidden layer. What do you have to add to `forward` and `backward`?
- Train on the moons with `noise=0.4`. How does the boundary and the test accuracy change?
- Plot the loss of the training and the test set for the moons and look for overfitting with 64 hidden neurons.

Then give a **3-minute demonstration**: show the forward pass shapes, the gradient check, the XOR training and the moons boundary.

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Improvement you chose:
- What you saw:
- What was the most surprising thing about opening the box:

> **Where next.** In Lab 38 you use a network that was made for pictures, a **convolutional neural network**, and train it to read handwritten digits on the GPU of the Jetson.

## 4. Check Task
Show your instructor. The instructor initials each line when it is done correctly.

| No. | Task to demonstrate | Initials |
|---|---|---|
| 1 | One neuron is calculated by hand and by code | |
| 2 | The forward pass of the network is run and the shapes are explained | |
| 3 | The numerical gradient is measured, and backpropagation passes the gradient check | |
| 4 | XOR is solved and the loss curve is shown | |
| 5 | The learning rate and the number of hidden neurons are compared | |
| 6 | The gradients agree with PyTorch autograd | |
| 7 | The moons are classified, with the decision boundaries for three sizes | |
| 8 | One improvement is done and the 3-minute presentation is given | |

## 5. Assessment Questions
Answer in one or two sentences.

**Your answers**

- 1. What does a neuron calculate? What is the activation function for?:
- 2. What are the shapes of X, W1, b1, W2 for 5 inputs, 8 hidden neurons and 3 examples?:
- 3. What does the loss measure? What is the loss of a coin-toss network?:
- 4. What is a gradient? Why do we move a weight against it?:
- 5. What is the learning rate? What happens when it is far too big?:
- 6. Why is a gradient check important? Why does a mistake in backpropagation not crash the program?:
- 7. Why can a single neuron not solve XOR, and a network with a hidden layer can?:
- 8. What does autograd do for you?:
- 9. Why does a network with too many hidden neurons overfit?:
- 10. Name one thing in a big network (like YOLO) that works the same as in this tiny network.:

## 6. Troubleshooting

| Symptom | Likely cause | What to try |
|---|---|---|
| The gradient check says WRONG | A formula in `backward` is mistyped, or `forward` was not called before | Compare each line with the table in the Background; call `net.forward(X)` before `net.backward(y)` |
| The loss does not go down | The learning rate is far too small or too big, or the weights update has the wrong sign | Use `lr = 0.5`; the update is `weight -= lr * gradient` |
| The loss is `nan` | The learning rate is far too big (the numbers explode) | Use a smaller learning rate |
| `ValueError: operands could not be broadcast` | A shape is wrong (for example `y` is a flat array) | Make `y` a column: shape (N, 1) |
| The XOR network is stuck at 0.69 | A bad starting point (for example only one hidden neuron) | Use 4 hidden neurons and another `seed` |
| The pictures do not appear | The plot cell was not run | Run the cell again |
| The notebook looks broken | You changed or deleted too much | Close it and run `import labview; labview.reset(37)` in a cell of a new notebook (File, New, Notebook). You get a fresh copy. |

## 7. Further Reading
- 3Blue1Brown, Neural networks (videos): https://www.3blue1brown.com/topics/neural-networks
- Karpathy, micrograd (a tiny autograd engine): https://github.com/karpathy/micrograd
- PyTorch, Autograd: https://pytorch.org/tutorials/beginner/blitz/autograd_tutorial.html